# Tennis Prediction Project

**Introduction**: This projects builds a prediction model with the aim to predict the winner of a tennis match.

### Clean dataset

#### Datasets

The main datasets used for this project were downloaded from [tennis-data.co.uk](http://tennis-data.co.uk), covering the years 2000 to 2026. Each file is stored locally as `data/raw/<year>.xlsx`.

In [89]:
'''Load all datasets from 2000 to 2026'''

from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None) # show all columns when displaying a dataframe

RAW_DIR = Path("../data/raw")

datasets = {} # {2000: df_2000, ...}

for file in sorted(RAW_DIR.glob("*.xlsx")):
    if file.stem.isdigit():
        year = int(file.stem)
        datasets[year] = pd.read_excel(file)

datasets[2005].head()

,ATP,Location,Tournament,Date,Series,Court,Surface,Round,Best of,Winner,Loser,WRank,LRank,WPts,LPts,W1,L1,W2,L2,W3,L3,W4,L4,W5,L5,Wsets,Lsets,Comment,B365W,B365L,CBW,CBL,EXW,EXL,IWW,IWL,PSW,PSL
0,1,Adelaide,Next Generation Hardcourts,2005-01-03,International,Outdoor,Hard,1st Round,3,Saulnier C.,Baccanello P.,53,324.0,NaN,NaN,6.0,2.0,7,6,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.0,Completed,1.286,3.250,1.25,3.7,1.3,3.35,1.30,2.70,1.305,3.780
1,1,Adelaide,Next Generation Hardcourts,2005-01-03,International,Outdoor,Hard,1st Round,3,Enqvist T.,Sluiter R.,72,82.0,NaN,NaN,6.0,3.0,6,1,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.0,Completed,1.833,1.833,1.85,1.9,1.8,1.95,1.75,1.75,1.990,1.840
2,1,Adelaide,Next Generation Hardcourts,2005-01-03,International,Outdoor,Hard,1st Round,3,Melzer J.,Berdych T.,39,45.0,NaN,NaN,6.0,4.0,4,6,7,6,NaN,NaN,NaN,NaN,2.0,1.0,Completed,1.800,1.909,1.75,2.0,1.9,1.85,1.85,1.65,1.901,1.917
3,1,Adelaide,Next Generation Hardcourts,2005-01-03,International,Outdoor,Hard,1st Round,3,Rochus O.,Dupuis A.,66,79.0,NaN,NaN,6.0,3.0,3,6,6,1,NaN,NaN,NaN,NaN,2.0,1.0,Completed,1.667,2.100,1.58,2.3,1.6,2.25,1.55,2.00,1.621,2.410
4,1,Adelaide,Next Generation Hardcourts,2005-01-03,International,Outdoor,Hard,1st Round,3,Mayer F.,Arthurs W.,35,101.0,NaN,NaN,6.0,4.0,3,6,7,5,NaN,NaN,NaN,NaN,2.0,1.0,Completed,1.615,2.200,1.75,2.0,1.8,1.95,1.55,2.00,1.787,2.070


In [90]:
'''Show the number of matches per year'''
matches_by_year = {
    year: len(df)
    for year, df in datasets.items()
}

result = pd.DataFrame(matches_by_year, index=["matches"])
display(result)

avg_matches = sum(len(df) for df in datasets.values()) / len(datasets)
max_year = max(datasets, key=lambda y: len(datasets[y]))
min_year = min(datasets, key=lambda y: len(datasets[y]))

print(f"Average matches per year: {avg_matches:.2f}")
print(f"Max matches: {len(datasets[max_year])} in {max_year}")
print(f"Min matches: {len(datasets[min_year])} in {min_year}")

,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
matches,2963,2963,2854,2861,2877,2909,2909,2806,2707,2731,2679,2675,2607,2631,2600,2630,2626,2633,2637,2610,1267,2489,2632,2703,2703,2644,1786


Average matches per year: 2634.52
Max matches: 2963 in 2000
Min matches: 1267 in 2020


Analyzing the datasets, we notice that the columns are not consistent across all years. In particular, the columns `WPts, LPts` are available only starting from **2005-07-04**.

In [91]:
# Remove datasets before year

split_year = 2005

datasets = {
    year: df
    for year, df in datasets.items()
    if year >= split_year
}

# For split_year, keep only rows from split_year-07-04 onwards
datasets[split_year]["Date"] = pd.to_datetime(datasets[split_year]["Date"])

datasets[split_year] = datasets[split_year][
    datasets[split_year]["Date"] >= f"{split_year}-07-04"
]

datasets[split_year].head()

,ATP,Location,Tournament,Date,Series,Court,Surface,Round,Best of,Winner,Loser,WRank,LRank,WPts,LPts,W1,L1,W2,L2,W3,L3,W4,L4,W5,L5,Wsets,Lsets,Comment,B365W,B365L,CBW,CBL,EXW,EXL,IWW,IWL,PSW,PSL
1676,37,Bastad,Swedish Open,2005-07-04,International,Outdoor,Clay,1st Round,3,Robredo T.,Tabara M.,20,112.0,1425.0,381.0,7.0,5.0,6,0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.0,Completed,1.10,6.00,1.08,7.25,1.10,6.75,1.10,5.0,1.111,8.750
1677,37,Bastad,Swedish Open,2005-07-04,International,Outdoor,Clay,1st Round,3,Vinciguerra A.,Ryderstedt M.,917,132.0,9.0,326.0,6.0,3.0,6,1,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.0,Completed,2.10,1.66,NaN,NaN,1.90,1.85,2.50,1.4,2.100,1.833
1678,37,Bastad,Swedish Open,2005-07-05,International,Outdoor,Clay,1st Round,3,Youzhny M.,Haehnel J.,27,109.0,1200.0,391.0,6.0,1.0,6,1,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.0,Completed,1.19,4.00,1.25,3.85,1.20,4.25,1.25,3.2,1.258,4.480
1679,37,Bastad,Swedish Open,2005-07-05,International,Outdoor,Clay,1st Round,3,Ferrero J.C.,Dlouhy L.,31,136.0,1150.0,315.0,6.0,4.0,6,1,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.0,Completed,1.07,7.00,1.06,8.50,1.05,8.50,1.10,5.0,NaN,NaN
1680,37,Bastad,Swedish Open,2005-07-05,International,Outdoor,Clay,1st Round,3,Berdych T.,Kim K.,42,71.0,811.0,541.0,6.0,2.0,6,3,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.0,Completed,1.19,4.00,1.25,3.85,1.25,3.90,1.25,3.2,1.312,3.910


Let's see what are the common columns across all datasets, starting from **2005-07-04**.

In [92]:
# Get the columns/features that are present in every remaining dataset
common_features = set.intersection(
    *[set(df.columns) for df in datasets.values()]
)

# Sort them to make the output easier to read
common_features = sorted(common_features)

print(f"Common features in all filtered datasets: {len(common_features)}")
print(common_features)

Common features in all filtered datasets: 30
['ATP', 'B365L', 'B365W', 'Best of', 'Comment', 'Court', 'Date', 'L1', 'L2', 'L3', 'L4', 'L5', 'LPts', 'LRank', 'Location', 'Loser', 'Lsets', 'Round', 'Series', 'Surface', 'Tournament', 'W1', 'W2', 'W3', 'W4', 'W5', 'WPts', 'WRank', 'Winner', 'Wsets']


Then, we can take a closer look at the feature types and missing values.

In [93]:
# Concatenate all datasets into a single DataFrame
all_matches = pd.concat(datasets.values(), ignore_index=True)

# Consider only common features
all_matches = all_matches[common_features]

summary = pd.DataFrame({
    "feature": all_matches.columns,
    "dtype": all_matches.dtypes.astype(str),
    "nulls": all_matches.isna().sum(),
    "unique": all_matches.nunique()
}).sort_values("nulls", ascending=False)

print(f"Shape: {all_matches.shape}")
print(summary.to_string(index=False))
print(f"dtypes: {all_matches.dtypes.value_counts().to_dict()}")

Shape: (54938, 30)
   feature          dtype  nulls  unique
        W5         object  52930      23
        L5         object  52907      22
        L4         object  49666       9
        W4         object  49657       9
        L3         object  28954      10
        W3         object  28953      10
        L2         object    831       9
        W2         object    829      10
     B365W         object    530     131
     B365L        float64    508     137
        W1         object    341       9
        L1         object    339      10
     Lsets        float64    338       3
     Wsets        float64    336       4
      LPts        float64    110    3199
     LRank        float64    110     946
     WRank        float64     19     607
      WPts        float64     17    3212
   Best of        float64     15       2
    Winner            str      0     990
       ATP          int64      0      68
Tournament            str      0     209
   Surface            str      0      

Let's check if there are any duplicates in the dataset.

In [94]:
keys = ["Date", "Winner", "Loser"]
n_duplicates = all_matches.duplicated(subset=keys, keep=False).sum()
print(f"Duplicate matches found: {n_duplicates}")

all_matches = all_matches.drop_duplicates(subset=keys, keep="first")
print(f"Shape after removing duplicates: {all_matches.shape}")

Duplicate matches found: 0
Shape after removing duplicates: (54938, 30)


We keep the most relevant features for our model, that will be used for feature engineering and training a performant model.

In [95]:
raw_features: list[str] = [
    "Date",
    "Series", "Court", "Surface", "Round", "Tournament", "Location",
    "Best of",
    "Winner", "Loser", "WRank", "LRank", "WPts", "LPts",
    "W1", "L1", "W2", "L2", "W3", "L3", "W4", "L4", "W5", "L5",
    "B365W","B365L",
    "Wsets", "Lsets",
]

### Merge the datasets
We merge the cleaned datasets into a single dataset, which will be used for further analysis and modeling.

In [96]:
all_matches = all_matches[raw_features].copy()
all_matches["Date"] = pd.to_datetime(all_matches["Date"], errors="raise")
all_matches = all_matches.sort_values("Date", kind="stable").reset_index(drop=True)

# Columns that should be numeric but are currently object
object_cols = ["W1", "L1", "W2", "L2", "W3", "L3", "W4", "L4", "W5", "L5", "B365W", "B365L"]
all_matches[object_cols] = all_matches[object_cols].apply(
    pd.to_numeric,
    errors="coerce" # coerce invalid parsing to NaN
)

print(all_matches.describe(), end="\n\n")

                             Date       Best of         WRank         LRank  \
count                       54938  54923.000000  54919.000000  54828.000000   
mean   2015-10-27 10:13:05.030398      3.384174     58.714470     90.688043   
min           2005-07-04 00:00:00      3.000000      1.000000      1.000000   
25%           2010-06-07 00:00:00      3.000000     16.000000     35.000000   
50%           2015-08-06 00:00:00      3.000000     41.000000     64.000000   
75%           2021-06-03 00:00:00      3.000000     77.000000    103.000000   
max           2026-08-03 00:00:00      5.000000   1890.000000   4915.000000   
std                           NaN      0.787890     73.054304    120.719069   

               WPts          LPts            W1            L1            W2  \
count  54921.000000  54828.000000  54595.000000  54598.000000  54105.000000   
mean    1867.796926   1108.863117      5.801319      4.112239      5.781647   
min        1.000000      1.000000      0.000000    

## Tennis players dataset

Another dataset we will use for our analysis is the atp tennis players dataset, with information about the players

In [97]:
RAW_TENNIS_PLAYERS_PATH = Path("../data/raw/atp_players.csv")

players_df = pd.read_csv(RAW_TENNIS_PLAYERS_PATH)

print(players_df.head())

   player_id name_first name_last hand         dob  ioc  height wikidata_id
0     100001    Gardnar    Mulloy    R  19131122.0  USA   185.0      Q54544
1     100002     Pancho    Segura    R  19210620.0  ECU   168.0      Q54581
2     100003      Frank   Sedgman    R  19271002.0  AUS   180.0     Q962049
3     100004   Giuseppe     Merlo    R  19271011.0  ITA     NaN    Q1258752
4     100005    Richard  Gonzalez    R  19280509.0  USA   188.0      Q53554


/var/folders/80/fvvpvft941v05nq1pl1gj2hc0000gn/T/ipykernel_69128/709771620.py:3: DtypeWarning: Columns (0: wikidata_id) have mixed types. Specify dtype option on import or set low_memory=False.
  players_df = pd.read_csv(RAW_TENNIS_PLAYERS_PATH)


#### Cleaning the dataset

Remove players with missing first or last names

In [98]:

print(players_df.shape[0])
players_df = players_df.dropna(subset=["name_first", "name_last", "dob", "ioc"])
print(players_df.shape[0])


65989
47537


Uniform playes full_name to match the format used in the matches dataset

In [99]:
import re
import unicodedata

def strip_accents(value):
    # Match files do not reliably keep accents, so use an ASCII-safe form.
    return unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode("utf-8")

def clean_name(value):
    # Use the same compact format for player metadata and match names.
    value = strip_accents(value).replace("-", " ")
    value = re.sub(r"\s+", " ", value).strip()
    return value

def build_initial_variants(first_name):
    # Match data usually uses one initial, but some compound names may use all initials.
    parts = [part for part in re.split(r"[-\s]+", clean_name(first_name)) if part]
    if not parts:
        return []

    variants = {f"{parts[0][0]}."}
    if len(parts) > 1:
        variants.add("".join(f"{part[0]}." for part in parts))

    return sorted(variants)

# Clean match player names once, before saving the raw combined/split datasets.
all_matches["Winner"] = all_matches["Winner"].apply(clean_name)
all_matches["Loser"] = all_matches["Loser"].apply(clean_name)

# Build match-compatible player keys, e.g. "Jannik Sinner" -> "Sinner J.".
players_df["name_first"] = players_df["name_first"].apply(clean_name)
players_df["name_last"] = players_df["name_last"].apply(clean_name)

player_rows = []
for row in players_df.itertuples(index=False):
    # # Expand each player into multiple rows for different initial variants to match all formats in the match dataset
    for initials in build_initial_variants(row.name_first):
        player_rows.append({
            "player_key": clean_name(f"{row.name_last} {initials}"),
            "ioc": row.ioc,
            "dob": row.dob,
            "hand": row.hand,
            "height": row.height,
        })

players_df = pd.DataFrame(player_rows)
players_df.head()


,player_key,ioc,dob,hand,height
0,Mulloy G.,USA,19131122.0,R,185.0
1,Segura P.,ECU,19210620.0,R,168.0
2,Sedgman F.,AUS,19271002.0,R,180.0
3,Merlo G.,ITA,19271011.0,R,NaN
4,Gonzalez R.,USA,19280509.0,R,188.0


Check for duplicates in the dataset

Let's check how many duplicated names we are actually using in the matches dataset.

In [100]:
duplicated_names = players_df.duplicated(subset=["player_key"], keep=False)
print("Players with duplicated match keys:", duplicated_names.sum())


Players with duplicated match keys: 7272


Firstly, we remove the duplicates with less information

In [101]:
# Keep duplicated match keys instead of dropping them.
# Names like "Ruud C." can refer to different players across eras.
# Feature engineering will disambiguate candidates using match date and plausible age.
players_df["info_count"] = players_df[["dob", "ioc", "hand", "height"]].notna().sum(axis=1)

players_df = (
    players_df
    .sort_values(["player_key", "info_count"], ascending=[True, False])
    .drop(columns="info_count")
    .copy()
)


In [102]:
duplicated_names = players_df.duplicated(subset=["player_key"], keep=False)
print("Rows with duplicated match keys kept for age disambiguation:", duplicated_names.sum())


Rows with duplicated match keys kept for age disambiguation: 7272


Save the cleaned players dataset to a new `.xlsx` file with only useful features.

In [103]:
players_df.notnull().sum()

player_key    52259
ioc           52259
dob           52259
hand          52253
height         4403
dtype: int64

In [104]:
TENNIS_PLAYERS_PATH = Path("../data/cleaned/tennis_players.xlsx")

players_features = [
    "player_key",
    "ioc",
    "dob",
    "hand",
]

players_df = players_df[players_features].copy()

# Convert YYYYMMDD values to real dates. Invalid dates become NaT and are ignored later.
players_df["dob"] = pd.to_datetime(
    players_df["dob"].astype("Int64").astype(str),
    format="%Y%m%d",
    errors="coerce"
)

players_df = players_df.dropna(subset=["player_key", "dob"]).copy()

TENNIS_PLAYERS_PATH.parent.mkdir(parents=True, exist_ok=True)
players_df.to_excel(TENNIS_PLAYERS_PATH, index=False)


Save the the matches datasets.

In [105]:
CLEANED_PATH = Path("../data/cleaned/tennis_matches.xlsx")

CLEANED_PATH.parent.mkdir(parents=True, exist_ok=True)
all_matches.to_excel(CLEANED_PATH, index=False)